# ERGM micro demo (Gemma 4 → train → plot)

Same spirit as **microGPT / nanoGPT**: a small config block, one training loop, and a loss curve — but data are **(s, a, s_next)** vectors for the geometric world model, not text tokens.

## Local Gemma 4 + Google Colab workflow

**Gemma 4 stays on your machine** (Ollama). Colab cannot see `localhost:11434` on your laptop unless you expose it (e.g. tunnel). Recommended flow:

1. **On your laptop:** generate data with Ollama + Gemma 4:
   `python -m ergm.generate_training_data_ollama --out data/transitions_gemma4`
2. **Upload** `transitions_gemma4.npz` to Colab (Files sidebar).
3. In the Colab setup cell below, set `os.environ["ERGM_NPZ"] = "/content/transitions_gemma4.npz"` (or your path).
4. Run the rest of the notebook to **train** the GeometricReasoner on GPU/CPU in Colab.

**Data sources (in the loader, in order):**
1. Load `npz` from `ERGM_NPZ` env or `data/transitions_gemma4.npz`.
2. Else call Ollama (only works if the notebook can reach your Ollama URL).
3. Else mock simulator fallback.

Prerequisites on laptop for generation: `ollama serve` and `ollama pull gemma4` (or your tag, e.g. `gemma4:4b`).

In [ ]:
# Colab: install deps; clone repo if needed; point to uploaded .npz from local Gemma/Ollama.
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q torch numpy matplotlib

# After uploading transitions_gemma4.npz from your laptop, uncomment and fix path:
# os.environ["ERGM_NPZ"] = "/content/transitions_gemma4.npz"

_repo = os.environ.get("ERGM_REPO_URL", "").strip()
if IN_COLAB and _repo and not (Path.cwd() / "ergm").is_dir():
    dest = Path("/content/cog2")
    if not dest.is_dir():
        subprocess.run(["git", "clone", _repo, str(dest)], check=True)
    os.chdir(dest)
    sys.path.insert(0, str(dest))
    print("Cloned and cwd:", dest)
elif IN_COLAB:
    print("Colab: set ERGM_REPO_URL to git clone, or open notebook from a cloned repo; set ERGM_NPZ for uploaded .npz")

In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
import urllib.error
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Repo root on sys.path (works if cwd is repo root or notebooks/)
_here = Path.cwd().resolve()
REPO_ROOT = _here if (_here / "ergm").is_dir() else _here.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ergm.constants import LATENT_DIM_LIGHT
from ergm.environment import get_initial_state, simulate_action_and_observe
from ergm.generate_training_data_ollama import (
    CONFIG as OLLAMA_CONFIG,
    SYSTEM_PROMPT,
    build_user_prompt,
)
from ergm.model import GeometricReasoner
from ergm.ollama_client import ollama_chat
from ergm.parse_llm_transitions import parse_transition_array, transitions_to_numpy
from ergm.tool_adapter import ToolAdapter
from ergm.training import geometric_prediction_loss, train_step

device = torch.device("cpu")
print("device:", device, "| REPO_ROOT:", REPO_ROOT)

In [ ]:
# ---- micro-style knobs (edit like microGPT config) ----
CONFIG = {
    "latent_dim": LATENT_DIM_LIGHT,  # use ergm.constants.LATENT_DIM for 512-wide run
    "action_dim": 64,
    "raw_dim": 8,
    "batch_size": 32,
    "max_steps": 200,
    "log_interval": 20,
    "lr": 3e-3,
    "npz_path": Path(os.environ["ERGM_NPZ"])
    if os.environ.get("ERGM_NPZ", "").strip()
    else (REPO_ROOT / "data" / "transitions_gemma4.npz"),
    "ollama_n": 8,  # small = faster first response for Gemma 4 on CPU
    "ollama_model": OLLAMA_CONFIG["model"],
    "ollama_host": OLLAMA_CONFIG["ollama_base_url"],
    "ollama_timeout_s": float(os.environ.get("OLLAMA_TIMEOUT", "600")),
}
CONFIG

In [ ]:
def load_or_make_transitions() -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    p = Path(CONFIG["npz_path"])
    if p.is_file():
        z = np.load(p)
        s, a, sn = z["s"], z["a"], z["s_next"]
        print(f"Loaded {s.shape[0]} transitions from {p}")
        return s.astype(np.float32), a.astype(np.float32), sn.astype(np.float32)

    n = int(CONFIG["ollama_n"])
    raw_dim = CONFIG["raw_dim"]
    sys_prompt = SYSTEM_PROMPT.format(raw_dim=raw_dim, count=n)
    user = build_user_prompt(raw_dim, n, 0, OLLAMA_CONFIG["temperature_hint"])
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": user},
    ]
    try:
        t0 = time.perf_counter()
        content = ollama_chat(
            CONFIG["ollama_host"],
            CONFIG["ollama_model"],
            messages,
            timeout_s=CONFIG["ollama_timeout_s"],
        )
        rows = parse_transition_array(content, raw_dim=raw_dim)
        s, a, sn = transitions_to_numpy(rows[:n])
        print(f"Gemma4 (Ollama) returned {len(rows)} rows in {time.perf_counter() - t0:.1f}s")
        return s, a, sn
    except (urllib.error.URLError, TimeoutError, ValueError, json.JSONDecodeError) as e:
        print("Ollama / parse failed:", e, "— using mock simulator data.")

    rows_list = []
    for i in range(max(n, 128)):
        s0 = get_initial_state(seed=i)
        a_np = np.random.default_rng(i).standard_normal(raw_dim)
        snp = simulate_action_and_observe(s0, a_np.astype(np.float64))
        rows_list.append({"s": s0.tolist(), "a": a_np.tolist(), "s_next": snp.tolist()})
    s, a, sn = transitions_to_numpy(rows_list)
    print(f"Mock simulator: {s.shape[0]} transitions")
    return s, a, sn


s_np, a_np, sn_np = load_or_make_transitions()
assert s_np.shape[1] == CONFIG["raw_dim"]
s_np.shape, a_np.shape, sn_np.shape

In [ ]:
D = CONFIG["latent_dim"]
raw_dim = CONFIG["raw_dim"]
action_dim = CONFIG["action_dim"]

tool_adapter = ToolAdapter(raw_dim=raw_dim, latent_dim=D).to(device)
reasoner = GeometricReasoner(latent_dim=D, action_dim=action_dim).to(device)
action_encoder = nn.Linear(raw_dim, action_dim, bias=False).to(device)

s_t_all = tool_adapter(torch.from_numpy(s_np).to(device))
target_all = tool_adapter(torch.from_numpy(sn_np).to(device))
a_t_all = action_encoder(torch.from_numpy(a_np).to(device))

N = s_t_all.shape[0]
perm = torch.randperm(N)
n_val = max(1, N // 5)
val_idx = perm[:n_val]
train_idx = perm[n_val:]
if train_idx.numel() == 0:
    train_idx = perm

opt = optim.Adam(
    list(reasoner.parameters()) + list(tool_adapter.parameters()) + list(action_encoder.parameters()),
    lr=CONFIG["lr"],
)

losses: list[float] = []
bs = min(CONFIG["batch_size"], max(1, train_idx.numel()))

for step in range(CONFIG["max_steps"]):
    idx = train_idx[torch.randint(0, train_idx.numel(), (bs,))]
    s_b = s_t_all[idx]
    a_b = a_t_all[idx]
    t_b = target_all[idx]
    L = train_step(reasoner, opt, s_b, a_b, t_b)
    losses.append(L)
    if (step + 1) % CONFIG["log_interval"] == 0:
        with torch.no_grad():
            vidx = val_idx
            val_mse = geometric_prediction_loss(reasoner(s_t_all[vidx], a_t_all[vidx]), target_all[vidx]).item()
        print(f"step {step+1}/{CONFIG['max_steps']}  train_mse={L:.6f}  val_mse={val_mse:.6f}")

print("done. final train losses (last 5):", losses[-5:])

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses, color="#2ecc71", lw=1.2, label="train MSE (batch)")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("ERGM GeometricReasoner — micro-style training (Gemma4 data or fallback)")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()